# EDA

In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [ ]:
print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [ ]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [ ]:
train.answer.value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [ ]:
train["prompt"].str.len().describe()

count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64

# milstone1

In [ ]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    TfidfVectorizer
)

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [ ]:
freq = train["answer"].value_counts()

print(freq)

most_freq = freq.max()
least_freq = freq.min()

print("ANSWER:", most_freq + least_freq)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
ANSWER: 814


In [ ]:
vocab = set()

for text in train["prompt"]:

    text = text.lower()

    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    words = text.split()

    vocab.update(words)

print("ANSWER:", len(vocab))

ANSWER: 859


In [ ]:
row1 = train.loc[
    train["id"] == 1,
    "prompt"
].iloc[0]

clean = row1.lower()

clean = clean.translate(
    str.maketrans(
        "",
        "",
        string.punctuation
    )
)

words = clean.split()

filtered = [
    w
    for w in words
    if w not in ENGLISH_STOP_WORDS
]

print(filtered)

print("ANSWER:", len(filtered))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
ANSWER: 13


In [ ]:
all_text = []

for _, row in train.iterrows():

    combined = " ".join([
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ])

    all_text.append(combined)

vectorizer = TfidfVectorizer(
    stop_words="english"
)

X = vectorizer.fit_transform(all_text)

print("ANSWER:", len(vectorizer.get_feature_names_out()))

ANSWER: 2762


In [ ]:
row1 = train.loc[
    train["id"] == 1
].iloc[0]

prompt_vec = vectorizer.transform(
    [row1["prompt"]]
)

A_vec = vectorizer.transform(
    [row1["A"]]
)

score = cosine_similarity(
    prompt_vec,
    A_vec
)[0][0]

print(round(score,4))

0.272


In [ ]:
correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform(
        [row["prompt"]]
    )

    scores = {}

    for option in ["A","B","C","D","E"]:

        option_vec = vectorizer.transform(
            [row[option]]
        )

        score = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        scores[option] = score

    pred = max(
        scores,
        key=scores.get
    )

    if pred == row["answer"]:
        correct += 1

accuracy = correct / len(train) * 100

print(accuracy)

13.55


In [ ]:
1

1

In [ ]:
0.5

0.5

In [ ]:
freq = train["answer"].value_counts()

top3 = list(freq.index[:3])

print(top3)

def apk(actual,predicted,k=3):

    for i,p in enumerate(predicted[:k]):
        if p == actual:
            return 1/(i+1)

    return 0

scores = []

for actual in train["answer"]:
    scores.append(apk(actual,top3))

print(np.mean(scores))

['B', 'C', 'A']
0.42125


In [ ]:
def apk(actual,predicted,k=3):

    for i,p in enumerate(predicted[:k]):
        if p == actual:
            return 1/(i+1)

    return 0

scores = []

for _,row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    sim_scores = {}

    for option in ["A","B","C","D","E"]:

        option_vec = vectorizer.transform([row[option]])

        sim_scores[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    ranked = sorted(
        sim_scores,
        key=sim_scores.get,
        reverse=True
    )

    scores.append(
        apk(
            row["answer"],
            ranked
        )
    )

print(np.mean(scores))

0.2961666666666667
